## Example 1: FastAI Core Modules and Components

In [ ]:
import fastai
from fastai.data.all import *
from fastai.tabular.all import *
import inspect
print('Example 1: FastAI Core Modules and Components\n')
print(f'FastAI version: {fastai.__version__}')
print('\nMain fastai modules:')
for mod in sorted([m for m in dir(fastai) if not m.startswith('_')])[:10]:
    print(f'  * {mod}')
print('\n\nDataBlock API - Key components:')
print(f'  * DataBlock: {DataBlock}')
print(f'  * DataLoaders: {DataLoaders}')
print(f'  * TransformBlock: {TransformBlock}')
print('\n\nTabular learner functions:')
for func in sorted([f for f in dir(fastai.tabular.learner) if 'learn' in f.lower() or 'tabular' in f.lower()])[:10]:
    print(f'  * {func}')
print('\n\nCore data transforms:')
for t in ['Categorify','Normalize','FillMissing','RandomSplitter']:
    print(f'  * {t}: {globals().get(t)}')
print('\nFastAI provides high-level APIs for deep learning tasks')

## Example 2: Creating synthetic tabular data and DataLoaders

In [ ]:
# Example 2: Creating synthetic tabular data and DataLoaders
from fastai.tabular.all import *
import pandas as pd
import numpy as np

print("Example 2: FastAI Tabular Data and DataLoaders\n")

# Create synthetic tabular dataset
print("Creating synthetic tabular dataset:")
np.random.seed(42)
n_samples = 1000
df = pd.DataFrame({
    'age': np.random.randint(18, 80, n_samples),
    'income': np.random.uniform(20000, 150000, n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'experience': np.random.randint(0, 40, n_samples),
    'target': np.random.choice([0, 1], n_samples)
})
print(f"  Dataset shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

# Define categorical and continuous variables
print("\n\nDefining features:")
cat_names = ['education']
cont_names = ['age', 'income', 'experience']
print(f"  Categorical: {cat_names}")
print(f"  Continuous: {cont_names}")

# Create TabularDataLoaders
print("\nCreating TabularDataLoaders:")
splits = RandomSplitter(valid_pct=0.2)(range_of(df))
to = TabularPandas(df, procs=[Categorify, Normalize],
                   cat_names=cat_names,
                   cont_names=cont_names,
                   y_names='target',
                   splits=splits)
dls = to.dataloaders(bs=64)
print(f"  Training samples: {len(dls.train_ds)}")
print(f"  Validation samples: {len(dls.valid_ds)}")
print(f"  Batch size: {dls.bs}")

# Show a batch
print("\nSample batch structure:")
batch = dls.one_batch()
print(f"  Batch type: {type(batch)}")
print(f"  Batch length: {len(batch)}")
if len(batch) == 3:
    cat_x, cont_x, y = batch
    print(f"  Categorical features shape: {cat_x.shape}")
    print(f"  Continuous features shape: {cont_x.shape}")
    print(f"  Target shape: {y.shape}")

print("\nFastAI TabularDataLoaders handle preprocessing automatically")

## Example 3: Creating a simple tabular learner

In [ ]:
# Example 3: Creating a simple tabular learner
from fastai.tabular.all import *
import pandas as pd
import numpy as np

print("Example 3: FastAI Tabular Learner\n")

# Create synthetic dataset for classification
np.random.seed(123)
n = 500
df = pd.DataFrame({
    'feature1': np.random.randn(n),
    'feature2': np.random.randn(n),
    'feature3': np.random.randn(n),
    'category': np.random.choice(['A', 'B'], n),
    'label': np.random.choice(['Class0', 'Class1'], n)
})
print(f"Dataset: {len(df)} samples\n")

# Prepare data
cat_names = ['category']
cont_names = ['feature1', 'feature2', 'feature3']
splits = RandomSplitter(valid_pct=0.3)(range_of(df))

to = TabularPandas(df, procs=[Categorify, Normalize],
                   cat_names=cat_names,
                   cont_names=cont_names,
                   y_names='label',
                   splits=splits)
dls = to.dataloaders(bs=32)
print(f"DataLoaders created: {len(dls.train_ds)} train, {len(dls.valid_ds)} valid\n")

# Create tabular learner
print("Creating tabular learner:")
learn = tabular_learner(dls, metrics=accuracy, layers=[64, 32])
print(f"  Learner type: {type(learn)}")
print(f"  Model architecture: {learn.model.__class__.__name__}")
print(f"  Metrics: {[m.name for m in learn.metrics]}")
print(f"  Loss function: {learn.loss_func}")

# Show model summary
print("\nModel summary:")
print(f"  Number of parameters: {sum(p.numel() for p in learn.model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in learn.model.parameters() if p.requires_grad):,}")

# Show optimizer
print(f"\nOptimizer: {learn.opt.__class__.__name__}")

print("\nFastAI tabular_learner simplifies neural network creation")

## Example 4: Exploring fastai callbacks and training components

In [ ]:
# Example 4: Exploring fastai callbacks and training components
from fastai.tabular.all import *
from fastai.callback.all import *
import pandas as pd
import numpy as np

print("Example 4: FastAI Callbacks and Training Components\n")

# Show available callbacks
print("Available callback classes:")
callback_classes = [c for c in dir(fastai.callback.core) if 'Callback' in c and c[0].isupper()]
for cb in sorted(callback_classes)[:10]:
    print(f"  • {cb}")

# Show tracker callbacks
print("\nTracker callbacks:")
tracker_cbs = [c for c in dir(fastai.callback.tracker) if c[0].isupper() and not c.startswith('_')]
for cb in sorted(tracker_cbs)[:8]:
    print(f"  • {cb}")

# Create simple data and learner to demonstrate callbacks
np.random.seed(456)
n = 300
df = pd.DataFrame({
    'x1': np.random.randn(n),
    'x2': np.random.randn(n),
    'y': np.random.choice([0, 1], n)
})

splits = RandomSplitter(valid_pct=0.2)(range_of(df))
to = TabularPandas(df, procs=[Normalize],
                   cat_names=[],
                   cont_names=['x1', 'x2'],
                   y_names='y',
                   splits=splits)
dls = to.dataloaders(bs=32)
learn = tabular_learner(dls, metrics=accuracy, layers=[16])

print("\n\nLearner callbacks:")
for i, cb in enumerate(learn.cbs):
    print(f"  {i+1}. {cb.__class__.__name__}")

# Show learning rate finder
print("\nLearning rate finder available:")
print(f"  lr_find method: {hasattr(learn, 'lr_find')}")

# Show recorder callback info
print("\nRecorder callback:")
recorder = learn.recorder
print(f"  Type: {type(recorder)}")
print(f"  Training: {recorder.training}")

print("\nFastAI callbacks enable flexible training customization")

## Example 5: FastAI Transforms and Preprocessing

In [ ]:
from fastai.tabular.all import *
from fastai.data.transforms import *
import pandas as pd, numpy as np
print('Example 5: FastAI Transforms and Preprocessing\n')
print('Core data transforms:')
for t in ['Categorify','Normalize','FillMissing','IntToFloatTensor','Categorize']:
    obj = globals().get(t)
    if obj is not None: print(f'  * {t}: {obj}')
np.random.seed(0)
df = pd.DataFrame({'grp': np.random.choice(['a','b','c'],200),'x1': np.random.randn(200)*5+10,'x2': np.random.uniform(0,10,200),'y': np.random.choice([0,1],200)})
splits = RandomSplitter(valid_pct=0.2)(range_of(df))
to = TabularPandas(df, procs=[Categorify, Normalize], cat_names=['grp'], cont_names=['x1','x2'], y_names='y', splits=splits)
print('\nApplied procs [Categorify, Normalize]:')
print(f"  * grp categories: {to.classes['grp']}")
print(f"  * x1 normalized mean~0: {to.train.xs['x1'].mean():.3f}, std~1: {to.train.xs['x1'].std():.3f}")
cat = Categorize(vocab=['neg','pos'])
enc = cat('pos'); dec = cat.decode(enc)
print(f"\nCategorize round-trip: 'pos' -> {int(enc)} -> '{dec}'")
print('\nFastAI provides extensive preprocessing capabilities')